# 5. Results, figures and failures

Builds every table and figure for the write-up and picks the demo clips. No GPU needed. A few minutes.

In [ ]:
# Setup: run this cell first. On Colab it clones the repo, installs packages and mounts Google Drive.
REPO_URL = "https://github.com/armaan-khoja/clear-voice"
SMOKE = False  # True = a quick check on a handful of clips (minutes, not hours)

import os, subprocess, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/clear-voice"):
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/clear-voice"], check=True)
    os.chdir("/content/clear-voice")
    subprocess.run(["git", "pull", "-q"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/clearvoice"
    try:
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        print("No HF_TOKEN secret: downloads still work, uploading models will not.")
else:
    WORK = os.path.abspath("work")
if SMOKE:
    WORK = f"{WORK}/smoke"  # quick checks never overwrite real models or results
sys.path.insert(0, os.getcwd())
os.makedirs(f"{WORK}/results", exist_ok=True)
os.makedirs(f"{WORK}/models", exist_ok=True)
SPLITS = "splits"
LIMIT = 6 if SMOKE else None
print("Results and models go to", WORK)

In [ ]:
import torch
print(torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "No GPU: Runtime > Change runtime type > L4 GPU, then rerun the setup cell.")

In [ ]:
import glob, os
import pandas as pd
from IPython.display import Image, display
from clearvoice import plots
from clearvoice.pipeline import align_folds, read_preds
from clearvoice.report import headline, leakage, per_speaker, pick_demo_clips, worst_clips
R = f"{WORK}/results"
zs = {s: read_preds(f"{R}/preds_zeroshot_{s}.csv") for s in ["whisper-small", "whisper-large-v3"]}
ft = pd.concat([read_preds(f"{R}/preds_ft_hold{f}.csv") for f in ["A", "B"]
                if os.path.exists(f"{R}/preds_ft_hold{f}.csv")], ignore_index=True)
zs = align_folds(zs, ft)  # same speakers for every system
preds = {**zs, "Clear Voice": ft}
print("Clear Voice folds found:", sorted(ft.fold.unique()))

**Headline:** dysarthric speakers the model never heard, saying sentences it never heard (95% intervals from resampling speakers).

In [ ]:
h = headline(preds)
h

**Leakage table:** the same model gets better-looking scores on easier tests.

In [ ]:
lk = leakage(ft)
lk

**Per speaker** (unseen sentences), including control speakers.

In [ ]:
ps = per_speaker(preds)
ps.pivot_table(index=['speaker', 'severity'], columns='system', values='word_cer').round(3)

**Everyday English** (LibriSpeech test-clean, 200 clips): did fine-tuning hurt typical speech?

In [ ]:
general = pd.concat([pd.read_csv(p) for p in sorted(glob.glob(f"{R}/librispeech_*.csv"))], ignore_index=True)
general

**The 10 worst clips.** Listen to these and write down why each one failed.

In [ ]:
w = worst_clips(ft, n=10)
w[["row_id", "speaker", "severity", "prompt_type", "ref", "hyp", "cer"]]

**Save tables, figures and demo clips.**

In [ ]:
os.makedirs(f"{R}/figures", exist_ok=True)
plots.bar_systems(h, f"{R}/figures/headline.png")
plots.leakage(lk, f"{R}/figures/leakage.png")
plots.per_speaker(ps, f"{R}/figures/per_speaker.png")
if os.path.exists(f"{R}/personalise.csv"):
    plots.personalisation(pd.read_csv(f"{R}/personalise.csv"), f"{R}/figures/personalisation.png")
for name, df in {"headline": h, "leakage": lk, "per_speaker": ps, "worst_clips": w, "general_speech": general}.items():
    df.to_csv(f"{R}/{name}.csv", index=False)

demo = pick_demo_clips(ft, per_severity=5).rename(columns={"hyp": "Clear Voice"})
for s, df in zs.items():
    demo = demo.merge(df[["row_id", "hyp"]].rename(columns={"hyp": s}), on="row_id", how="left")
meta = pd.read_pickle(f"{WORK}/meta.pkl")  # saved by notebook 1: where each clip sits in the dataset files
demo = demo.merge(meta[["row_id", "file_idx", "row_in_file"]], on="row_id", how="left")
demo.to_csv(f"{R}/demo_clips.csv", index=False)

with open(f"{R}/summary.md", "w") as f:
    for name, df in {"Headline": h, "Leakage": lk, "Everyday English": general}.items():
        f.write(f"## {name}\n\n{df.round(3).to_markdown(index=False)}\n\n")
for png in sorted(glob.glob(f"{R}/figures/*.png")):
    display(Image(png))

**Publish the demo** (optional). Creates a public Hugging Face Space under your account with the app, these results and figures. Audio is not uploaded: the app fetches clips from the TORGO dataset. Needs your `HF_TOKEN` secret and both models uploaded from notebook 3.

In [ ]:
DEPLOY_SPACE = False  # True publishes the demo as a public Hugging Face Space
if DEPLOY_SPACE:
    import shutil, tempfile
    from huggingface_hub import HfApi, whoami
    user = whoami()["name"]
    space = f"{user}/clear-voice"
    stage = os.path.join(tempfile.mkdtemp(), "space")
    shutil.copytree("app", stage, ignore=shutil.ignore_patterns("__pycache__", "flagged"))
    shutil.copytree("clearvoice", f"{stage}/clearvoice", ignore=shutil.ignore_patterns("__pycache__"))
    shutil.copy(f"{R}/demo_clips.csv", stage)
    shutil.copy(f"{R}/summary.md", stage)
    shutil.copytree(f"{R}/figures", f"{stage}/figures")
    api = HfApi()
    api.create_repo(space, repo_type="space", space_sdk="gradio", exist_ok=True)
    api.add_space_variable(space, "CV_MODEL_A", f"{user}/clear-voice-holda")
    api.upload_folder(folder_path=stage, repo_id=space, repo_type="space")
    print("Demo: https://huggingface.co/spaces/" + space)